<a href="https://colab.research.google.com/github/Brian342/Beans-Classification/blob/main/Bean_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install ultralytics -q
# !pip install pandas numpy Pillow opencv-python-headless -q

In [2]:
import os, shutil, ast, random, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import torch
from sklearn.model_selection import train_test_split
import yaml

In [7]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# BASE_DIR       = '/content/drive/MyDrive/BeansClassification'
# TRAIN_CSV      = f'{BASE_DIR}/Train.csv'
# TEST_CSV       = f'{BASE_DIR}/Test.csv'
# TRAIN_IMG_DIR = '/content/all_images/images'
# TEST_IMG_DIR  = '/content/all_images/images'
# YOLO_DIR      = '/content/yolo_dataset'

# print(f'Train CSV: {TRAIN_CSV}')
# print(f'Test  CSV: {TEST_CSV}')


Mounted at /content/drive
Train CSV: /content/drive/MyDrive/BeansClassification/Train.csv
Test  CSV: /content/drive/MyDrive/BeansClassification/Test.csv


In [13]:
BASE_DIR      = '/content/drive/MyDrive/BeansClassification'
TRAIN_CSV     = f'{BASE_DIR}/Train.csv'
TEST_CSV      = f'{BASE_DIR}/Test.csv'
TRAIN_IMG_DIR = f'{BASE_DIR}/images'
TEST_IMG_DIR  = f'{BASE_DIR}/images'
YOLO_DIR      = '/content/yolo_dataset'

import pandas as pd
from pathlib import Path

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train rows: {len(train_df)}")
print(f"Test  rows: {len(test_df)}")
print(f"Train images folder exists: {Path(TRAIN_IMG_DIR).exists()}")
print(f"Files in images folder: {len(list(Path(TRAIN_IMG_DIR).iterdir()))}")
print()
print("Sample Image_IDs from Train.csv:")
print(train_df['Image_ID'].unique()[:5])
print()

sample_id = train_df['Image_ID'].iloc[0]
for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
    p = Path(TRAIN_IMG_DIR) / f'{sample_id}{ext}'
    if p.exists():
        print(f"Sample image found: {p}")
        break

Train rows: 12142
Test  rows: 352
Train images folder exists: True
Files in images folder: 381

Sample Image_IDs from Train.csv:
['zpnCucYXL5Ta' 'RoVsYaebTYkI' 'RQT1orUqN3Ur' 'a0AhtNcl8PU2' 'LggZwvNYR200']



In [5]:
# BASE_DIR      = '/content/drive/MyDrive/BeansClassification'
# TRAIN_CSV     = f'{BASE_DIR}/Train.csv'
# TEST_CSV      = f'{BASE_DIR}/Test.csv'
# TRAIN_IMG_DIR = '/content/all_images/images'
# TEST_IMG_DIR  = '/content/all_images/images'
# YOLO_DIR      = '/content/yolo_dataset'

# from pathlib import Path

# disk_ids  = {f.stem for f in Path(TRAIN_IMG_DIR).iterdir() if f.is_file()}
# csv_ids   = set(train_df['Image_ID'].unique())
# test_ids  = set(test_df['Image_ID'].unique())

# matched_train = csv_ids & disk_ids
# matched_test  = test_ids & disk_ids

# print(f"Total images on disk:       {len(disk_ids)}")
# print(f"Train images matched:       {len(matched_train)} / {len(csv_ids)}")
# print(f"Test  images matched:       {len(matched_test)} / {len(test_ids)}")
# print()

# if len(matched_train) == len(csv_ids):
#     print("All train images found! Ready to run Steps 4 → 5 → 6")
# elif len(matched_train) > 800:
#     print("Most train images found — good enough to train!")
# else:
#     print(f"Only {len(matched_train)} matched — something may still be off")

FileNotFoundError: [Errno 2] No such file or directory: '/content/all_images/images'

In [16]:
import shutil, os

if os.path.exists('/content/yolo_dataset'):
    shutil.rmtree('/content/yolo_dataset')
    print("Old YOLO dataset cleared")

from pathlib import Path

sample_id = train_df['Image_ID'].iloc[0]
img_path = Path(TRAIN_IMG_DIR) / f'{sample_id}.jpg'
print(f"Sample image found: {img_path.exists()} → {img_path}")

print(f"\nUnique train Image_IDs: {train_df['Image_ID'].nunique()}")
print(f"Total images in folder: {len(list(Path(TRAIN_IMG_DIR).iterdir()))}")
print(f"\nNote: {train_df['Image_ID'].nunique()} unique images in CSV")
print(f"      381 files in folder — some may be test images too")

Old YOLO dataset cleared
Sample image found: False → /content/drive/MyDrive/BeansClassification/images/zpnCucYXL5Ta.jpg

Unique train Image_IDs: 985
Total images in folder: 381

Note: 985 unique images in CSV
      381 files in folder — some may be test images too


In [17]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Train columns:", train_df.columns.tolist())
print()
print("Class distribution:")
print(train_df['class'].value_counts())
print()
print("Unique train images:", train_df["Image_ID"].nunique())
print("Unique test images:", test_df["Image_ID"].nunique())


Train shape: (12142, 6)
Test shape: (352, 3)

Train columns: ['Image_ID', 'image_width', 'image_height', 'bbox', 'points', 'class']

Class distribution:
class
Flower_closed    6003
Flower_open      4085
Plant_bean       2054
Name: count, dtype: int64

Unique train images: 985
Unique test images: 352


In [18]:
import os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

CLASS_MAP = {
    'Plant_bean':    0,
    'Flower_open':   1,
    'Flower_closed': 2
}
ID_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}

def parse_points(points_str):
    """Parse 'x1,y1;x2,y2;...' into list of (x, y) tuples."""
    pairs = points_str.strip().split(';')
    coords = []
    for p in pairs:
        x, y = p.strip().split(',')
        coords.append((float(x), float(y)))
    return coords

def normalize_polygon(points, img_w, img_h):
    """Normalize polygon coordinates to [0, 1] for YOLO format."""
    normalized = []
    for x, y in points:
        normalized.extend([x / img_w, y / img_h])
    return normalized

def create_yolo_dataset(df, img_src_dir, split_name, yolo_dir):
    img_out = Path(yolo_dir) / 'images' / split_name
    lbl_out = Path(yolo_dir) / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    grouped = df.groupby('Image_ID')
    skipped = 0

    for img_id, group in grouped:
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            candidate = Path(img_src_dir) / f'{img_id}{ext}'
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            continue

        dest_img = img_out / img_path.name
        if not dest_img.exists():
            shutil.copy(img_path, dest_img)

        img_w = group['image_width'].iloc[0]
        img_h = group['image_height'].iloc[0]
        label_lines = []

        for _, row in group.iterrows():
            cls_id = CLASS_MAP.get(row['class'])
            if cls_id is None:
                continue
            try:
                points = parse_points(row['points'])
                norm = normalize_polygon(points, img_w, img_h)
                coords_str = ' '.join(f'{v:.6f}' for v in norm)
                label_lines.append(f'{cls_id} {coords_str}')
            except Exception as e:
                continue

        lbl_file = lbl_out / f'{img_id}.txt'
        with open(lbl_file, 'w') as f:
            f.write('\n'.join(label_lines))

    print(f'[{split_name}] Done. Skipped {skipped} images not found on disk.')
    return img_out, lbl_out

unique_ids = train_df['Image_ID'].unique()
train_ids, val_ids = train_test_split(unique_ids, test_size=0.15, random_state=42)

train_split = train_df[train_df['Image_ID'].isin(train_ids)]
val_split   = train_df[train_df['Image_ID'].isin(val_ids)]

print(f'Train images: {len(train_ids)} | Val images: {len(val_ids)}')

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

create_yolo_dataset(train_split, TRAIN_IMG_DIR, 'train', YOLO_DIR)
create_yolo_dataset(val_split,   TRAIN_IMG_DIR, 'val',   YOLO_DIR)

print('YOLO dataset ready')

Train images: 837 | Val images: 148
[train] Done. Skipped 708 images not found on disk.
[val] Done. Skipped 129 images not found on disk.
YOLO dataset ready


In [19]:
# Create YOLO data.yaml
data_yaml = {
    'path': YOLO_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': ['Plant_bean', 'Flower_open', 'Flower_closed']
}

yaml_path = f'{YOLO_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print('data.yaml contents:')
print(open(yaml_path).read())

data.yaml contents:
names:
- Plant_bean
- Flower_open
- Flower_closed
nc: 3
path: /content/yolo_dataset
train: images/train
val: images/val



In [ ]:
# Train YOLOV8-seg Model
model = YOLO('yolov8m-seg.pt')

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Using device: {"GPU (cuda:0)" if device == 0 else "CPU"}')

if device == 'cpu':
    print('WARNING: Training on CPU will be very slow.')
    print('Go to Runtime → Change runtime type → T4 GPU')
# Train
results = model.train(
    data    = yaml_path,
    epochs  =100,
    imgsz   = 1024,
    batch   = 8,
    patience    = 15,
    device  = 0,
    project    = '/content/runs',
    name = 'bean_seg',
    exist_ok    = True,
# Augmentation
    hsv_h = .015,
    hsv_s = .7,
    hsv_v = .4,
    flipud = .5,
    fliplr = .5,
    mosaic = 1.0,
    degrees = 10.0,
    translate = .1,
    scale = .5
)
print("Training complete")
print("Best Model Saved at:", results.save_dir)

Using device: GPU (cuda:0)
New https://pypi.org/project/ultralytics/8.4.31 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, mult

In [21]:
# Evaluate on Validation set
best_model_path ="/content/runs/bean_seg/weights/best.pt"
model = YOLO(best_model_path)

# Validate
metrics = model.val(data=yaml_path, imgsz=640, iou=.50, conf=.25)
print("\n===== Validation metrics ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAp50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8m-seg summary (fused): 106 layers, 27,224,121 parameters, 0 gradients, 104.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3683.3±192.9 MB/s, size: 1941.2 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 8.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5s/it 3.0s
                   all         19        268      0.717      0.455      0.587      0.377      0.542      0.382      0.472      0.236
            Plant_bean         19         40      0.628      0.675      0.634      0.454      0.581      0.625      0.592       0.38
           Flower_open         14        104      0.655      0.529      0.609      0.411      0.524      0.423      0.514      0.231
         Flower_closed         18        1

In [22]:
# Generate Predictions on Test set
CONF_THRESHOLD =.25
IOU_THRESHOLD = .45

CLASS_NAMES = ["Plant_bean", "Flower_open", "Flower_closed"]

def get_test_image_path(img_id, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = Path(img_dir) / f"{img_id}{ext}"
        if  p.exists():
            return str(p)
    return None

submission_rows = []
test_ids = test_df["Image_ID"].unique()
missing = []

print(f"Running inference on {len(test_ids)} test Iimages...")

for i, img_id in enumerate(test_ids):
    img_path = get_test_image_path(img_id, TEST_IMG_DIR)

    if img_path is None:
        missing.append(img_id)
        submission_rows.append({
            'Image_ID': img_id,
            'class': '__background__',
            'confidence': .0,
            'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax':0
        })
        continue

    results = model.predict(
        source = img_path,
        imgsz = 640,
        conf = CONF_THRESHOLD,
        iou = IOU_THRESHOLD,
        verbose = False
    )

    result = results[0]
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        submission_rows.append({
            'Image_ID':   img_id,
            'class':      '__background__',
            'confidence': 0.0,
            'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax': 0
        })
        continue
    orig_h, orig_w = result.orig_shape

    for j in range(len(boxes)):
        conf = float(boxes.conf[j].cpu())
        cls_id = int(boxes.cls[j].cpu())
        cls_name = CLASS_NAMES[cls_id]

        if result.masks is not None and j < len(result.masks):
            mask = result.masks.data[j].cpu().numpy()
            mask_resized = cv2.resize(
                mask, (orig_w, orig_h), interpolation =cv2.INTER_NEAREST
            )
            ys, xs = np.where(mask_resized > .5)
            if len(xs) > 0 and len(ys) > 0:
                xmin, xmax = int(xs.min()), int(xs.max())
                ymin, ymax = int(ys.min()), int(ys.max())
            else:
                x1, y1, x2,y2 = boxes.xyxy[j].cpu().numpy()
                xmin, ymin, xmax, ymax = int(x1), int(y1), int(x2), int(y2)

        else:
            x1, y1, x2, y2 = boxes.xyxy[j].cpu().numpy()
            xmin, ymin, xmax, ymax = int(x1), int(y1), int(x2), int(y2)

        submission_rows.append({
            'Image_ID': img_id,
            'class': cls_name,
            'confidence': round(conf, 6),
            'ymin': ymin,
            'xmin': xmin,
            'ymax': ymax,
            'xmax': xmax
        })

    if (i + 1) % 50 == 0:
        print(f" {i +1}/{len(test_ids)} processed ...")
print(f"\n Inference done. Missing images: {len(missing)}")

Running inference on 352 test Iimages...

 Inference done. Missing images: 306


In [23]:
# Bulding and saving the submission csv
submission_df = pd.DataFrame(submission_rows, columns=[
    'Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax'
])
submission_df = submission_df.sort_values(
    ['Image_ID', "confidence"], ascending=[True, False]
).reset_index(drop=True)

print("Submission Sumaary")
print(f"Total rows: {len(submission_df)}")
print(f"Unique Image_IDs:   {submission_df["Image_ID"].nunique()}")
print(f"Expected images:    {len(test_ids)}")
print()
print('class distribution in predictions:')
print(submission_df['class'].value_counts())
print()
print('Sample rows: ')
display(submission_df.head(10))

submitted_ids = set(submission_df['Image_ID'].unique())
missing_from_submission = set(test_ids) - submitted_ids
if missing_from_submission:
    print(f"\n error: {len(missing_from_submission)} images missing from submission!")

    bg_rows = [{
        'Image_ID': img_id, 'class': "__background__",
        'confidence': .0, 'ymin': 0, 'xmin':0, 'ymax':0, "xmax":0
    } for img_id in missing_from_submission]
    sumission_df = pd.concat(
        [submission_df, pd.DataFrame(bg_rows)], ignore_index=True
    )
    print("Background rows added for missing images")
else:
    print("All test images are covered in submission!!")

SUBMISSION_PATH = f"{BASE_DIR}/submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to: {SUBMISSION_PATH}")

Submission Sumaary
Total rows: 757
Unique Image_IDs:   352
Expected images:    352

class distribution in predictions:
class
__background__    306
Flower_open       246
Plant_bean        125
Flower_closed      80
Name: count, dtype: int64

Sample rows: 


,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,04DZeTP47eXR,Flower_open,0.822848,2742,2385,2894,2473
1,04DZeTP47eXR,Flower_open,0.788166,2232,2181,2326,2256
2,04DZeTP47eXR,Flower_open,0.786971,2053,2359,2154,2486
3,04DZeTP47eXR,Flower_open,0.746447,2130,1722,2231,1823
4,04DZeTP47eXR,Flower_open,0.549561,396,2512,567,2715
5,04DZeTP47eXR,Flower_open,0.492259,957,1824,1032,1886
6,04DZeTP47eXR,Flower_open,0.451598,3507,1671,3620,1753
7,04DZeTP47eXR,Flower_open,0.407050,0,2895,140,2983
8,04DZeTP47eXR,Plant_bean,0.403345,2793,1435,4079,2919
9,04DZeTP47eXR,Plant_bean,0.329062,13,1097,3098,3059


All test images are covered in submission!!
Submission saved to: /content/drive/MyDrive/BeansClassification/submission.csv


In [ ]:
# Tune Confidence Threshold for better F!
from collections import defaultdict

def computer_iou_bbox(boxA, boxB):
    """
    Compute IoU between two [ymin, xmin, ymax, xmax] boxes
    """
    yA = max(boxA[0], boxB[0]); xA = max(boxA[1], boxB[1])
    yB = min(boxA[2], box[2]); xB = min(boxA[3], boxB[3])
    inter = max(0, yB - yA) * max(0, xB - xA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else .0

def f1_at_

In [ ]:
# Data observation
# Show case the dataset information
def data_info(x):
    """
    This function is used to display the dataset information like:
    - shape of dataset
    - Statistical Info
    - Column Datatype
    - Null Value sum
    - Dulplicate values
    ** Parameters: X representing the data

    ** Return: None
    """
    print("================================================")
    print("Number of Rows and Columns In the DataSet")
    row, col = x.shape
    print(f"Data_Rows: {row} Data_col: {col}")
    print()
    print("================================================")
    print("Dataset Columns")
    print(x.columns)
    print()
    print("================================================")
    print("Dataset types")
    print(x.dtypes)
    print()
    print("================================================")
    print("DataSet Stats Info")
    print(x.describe())
    print()
    print("================================================")
    print("Dataset Column's Datatype")
    print(x.info())
    print()
    print("================================================")
    print("Dataset Null Count")
    print(x.isna().sum())
    print()
    print("================================================")
    print("Dataset Duplicated Values")
    print(x.duplicated().unique())
    print()
    print("================================================")
    print("Display All the Missing Values In a Chart")
    print(msno.bar(x))

    return None

In [ ]:
data_info(df_train)

In [ ]:
df_train.head(5)

In [ ]:
df_train["class"].unique()